In [ ]:
!pip install -q -U google-genai pymongo pandas matplotlib


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.


In [ ]:
import google.genai
import pymongo
import pandas as pd
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

All libraries imported successfully!


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Gemini API key is missing!")

print("Gemini API key configured successfully!")



Gemini API key configured successfully!


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(
        timeout=30000
    )
)

# Updated to gemini-3.8-flash per user request and API suggestions
MODEL_NAME = "gemini-3.8-flash"

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [ ]:
MONGODB_URI = userdata.get("MONGODB_URL")

if not MONGODB_URI:
    raise ValueError("MongoDB connection string is missing!")

print("MongoDB credentials loaded successfully!")


MongoDB credentials loaded successfully!


In [ ]:
import urllib.parse
from pymongo import MongoClient
from pymongo.server_api import ServerApi

try:
    # Remove placeholder < and > if they are present
    cleaned_uri = MONGODB_URI.replace("<", "").replace(">", "")

    # Split scheme from the rest
    scheme, rest = cleaned_uri.split("://", 1)

    # Split userinfo and hostinfo from the rightmost '@' character
    # because host part contains no '@' but password might contain '@'
    userinfo, hostinfo = rest.rsplit("@", 1)

    # Split username and password from the first ':' character
    username, password = userinfo.split(":", 1)

    # Quote username and password to escape special characters securely
    quoted_username = urllib.parse.quote_plus(username)
    quoted_password = urllib.parse.quote_plus(password)

    # Reconstruct the correct URI
    escaped_uri = f"{scheme}://{quoted_username}:{quoted_password}@{hostinfo}"
except Exception as e:
    print("Could not automatically parse the URI. Falling back to original URI.")
    escaped_uri = MONGODB_URI

mongo_client = MongoClient(
    escaped_uri,
    server_api=ServerApi("1"),
    serverSelectionTimeoutMS=5000,
    connectTimeoutMS=5000,
    socketTimeoutMS=10000,
    timeoutMS=10000,
    retryWrites=True
)

# Test the database connection
mongo_client.admin.command("ping")

# Create database
db = mongo_client["GenAI_Monitoring"]

# Create collection
collection = db["api_usage_logs"]

print("MongoDB connected successfully!")
print("Database:", db.name)
print("Collection:", collection.name)

MongoDB connected successfully!
Database: GenAI_Monitoring
Collection: api_usage_logs


In [ ]:
import time
import uuid
import random
import pymongo

from datetime import datetime, timezone
from pymongo.errors import PyMongoError
# Gemini 2.5 Flash standard paid-tier pricing
# USD per 1 million tokens

INPUT_PRICE = 0.30
OUTPUT_PRICE = 2.50

def calculate_cost(input_tokens, output_tokens):

    input_cost = (
        input_tokens * INPUT_PRICE / 1_000_000
    )

    output_cost = (
        output_tokens * OUTPUT_PRICE / 1_000_000
    )

    total_cost = input_cost + output_cost

    return total_cost

input_tokens = 1000
output_tokens = 2000

cost = calculate_cost(
    input_tokens,
    output_tokens
)

print("Estimated cost: $", cost)

def retry_delay(attempt):

    delay = min(2 ** (attempt - 1), 16)

    return delay + random.uniform(0, 1)


def is_temporary_error(error):

    status_code = getattr(
        error, "status_code",
        getattr(error, "code", None)
    )

    if status_code in [408, 429, 500, 502, 503, 504]:
        return True

    error_name = type(error).__name__.lower()

    temporary_errors = [
        "timeout",
        "connection",
        "temporarily"
    ]

    return any(
        word in error_name
        for word in temporary_errors
    )



Estimated cost: $ 0.0053


In [ ]:
def save_log(record):

    for attempt in range(1, 4):

        try:

            with pymongo.timeout(10):

                collection.update_one(
                    {"_id": record["_id"]},
                    {"$setOnInsert": record},
                    upsert=True
                )

            print("Log saved in MongoDB!")
            return True

        except PyMongoError as error:

            print(
                f"MongoDB error: {error}"
            )

            if attempt == 3:
                print("Unable to save the log.")
                return False

            time.sleep(retry_delay(attempt))

    return False


In [ ]:
def ask_gemini(prompt):

    request_id = str(uuid.uuid4())

    MAX_RETRIES = 4

    for attempt in range(1, MAX_RETRIES + 1):

        start_time = time.perf_counter()

        input_tokens = 0
        output_tokens = 0
        total_tokens = 0

        response_text = ""
        error_message = None
        status = "failed"

        try:

            # Step 1: Send request to Gemini

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            # Step 2: Extract AI response

            response_text = response.text or ""

            # Step 3: Extract token information

            usage = response.usage_metadata

            if usage:

                input_tokens = (
                    usage.prompt_token_count or 0
                )

                output_tokens = (
                    usage.candidates_token_count or 0
                )

                total_tokens = (
                    usage.total_token_count or 0
                )

            # Step 4: Mark request as successful

            status = "success"

        except Exception as error:

            error_message = str(error)

            print(
                f"API attempt {attempt} failed: "
                f"{error_message}"
            )

        # Step 5: Measure latency

        latency = time.perf_counter() - start_time

        # Step 6: Calculate estimated cost

        estimated_cost = calculate_cost(
            input_tokens,
            output_tokens
        )

        # Step 7: Prepare MongoDB record

        log_record = {

            "_id": f"{request_id}_{attempt}",

            "request_id": request_id,

            "timestamp": datetime.now(timezone.utc),

            "model": MODEL_NAME,

            "prompt": prompt,

            "attempt": attempt,

            "input_tokens": input_tokens,

            "output_tokens": output_tokens,

            "total_tokens": total_tokens,

            "latency_seconds": round(latency, 3),

            "estimated_cost_usd": round(
                estimated_cost, 10
            ),

            "status": status,

            "error": error_message
        }

        # Step 8: Store the record in MongoDB

        save_log(log_record)

        # Step 9: Display the monitoring information

        print("\n========== API MONITORING ==========")

        print("Request ID:", request_id)

        print("Attempt:", attempt)

        print("Status:", status)

        print("Input Tokens:", input_tokens)

        print("Output Tokens:", output_tokens)

        print("Total Tokens:", total_tokens)

        print("Latency:", round(latency, 3), "seconds")

        print("Estimated Cost: $", estimated_cost)

        # Step 10: Return successful response

        if status == "success":

            print("\n========== GEMINI RESPONSE ==========")

            print(response_text)

            return response_text

        # Step 11: Retry only temporary failures

        if (
            attempt == MAX_RETRIES
            or not is_temporary_error(error)
        ):

            print("\nRequest failed permanently.")
            break

        wait_time = retry_delay(attempt)

        print(
            f"Retrying after {wait_time:.2f} seconds..."
        )

        time.sleep(wait_time)

    return None


In [ ]:
while True:

    prompt = input("\nEnter your question (or type exit): ")

    if prompt.lower() == "exit":
        print("Application stopped.")
        break

    if not prompt.strip():
        print("Please enter a valid question.")
        continue

    ask_gemini(prompt)



Enter your question (or type exit): Explain Generative AI
API attempt 1 failed: 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status': 'DEADLINE_EXCEEDED'}}
Log saved in MongoDB!

========== API MONITORING ==========
Request ID: 47968777-e100-4d21-bf9f-e5a5c4a656a3
Attempt: 1
Status: failed
Input Tokens: 0
Output Tokens: 0
Total Tokens: 0
Latency: 28.33 seconds
Estimated Cost: $ 0.0


NameError: name 'is_temporary_error' is not defined